# AMD-SD: does denoising change what a segmentation model finds?

Sections 8-10 of `explore_amd_sd.ipynb` measured contrast and CNR. Those are proxies. This notebook
tests what matters clinically: **does a segmentation model still find each lesion after denoising,
does it invent new ones, and how much does the measured lesion area change?**

1. **Data:** scans and masks decoded once and cached (`seg_cache/`).
2. **Train a U-Net** on the original training scans (checkpointed every epoch, resumable).
3. **Evaluate** the same model on every validation scan, once per condition: the original scan and
   each denoised version found in `denoised_cache/`.
4. **Compare** lesion detection, missed and invented lesions, and area error by lesion size.

**Adding a new denoiser later** (Noise2Void, a GAN, ...): save its outputs as
`denoised_cache/<method_name>/<scan>.png` (same size as the BM3D files, greyscale uint8) and rerun
from section 4. Nothing else changes.

**Time on an M-series Mac:** data cache ~2 min (first run only), training roughly 2-4 h for 40
epochs (the first epoch prints its duration: multiply by 40), evaluation ~10 min. Every stage resumes
from its checkpoint if interrupted: just run all cells again. To check everything works first, set
`QUICK_TEST = True` in the next cell (about 5 min), then set it back to `False`.

**Requires:** `torch numpy pillow scipy pandas matplotlib`.

In [ ]:
from pathlib import Path
import os, sys, time, json, subprocess
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path.cwd().parent if Path.cwd().name == "_analysis" else Path.cwd()
IMG_DIR = ROOT / "images"
DENOISED = Path("denoised_cache")
CACHE = Path("seg_cache"); CACHE.mkdir(exist_ok=True)
CKPT = Path("seg_checkpoints"); CKPT.mkdir(exist_ok=True)

QUICK_TEST = False   # True = tiny run to check everything works (few scans, 2 epochs)

W_PANEL = 570        # each PNG = scan (570 px) | mask (570 px)
W_USE = 558          # crop the last 12 columns: mask colour bleeds into them
H, PAD_H, PAD_W = 380, 384, 560   # U-Net needs sizes divisible by 16

CLASSES = ["background", "SRF", "IRF", "PED", "SHRM", "ISOS"]
LESIONS = ["SRF", "IRF", "PED", "SHRM"]           # ISOS is a layer: trained on, not scored as a lesion
LESION_IDS = [CLASSES.index(c) for c in LESIONS]
IGNORE = 255
MIN_LESION_PX = 20

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print("root:", ROOT, "| device:", DEVICE, "| quick test:", QUICK_TEST)

if sys.platform == "darwin":   # keep the Mac awake while this kernel runs
    subprocess.Popen(["caffeinate", "-i", "-w", str(os.getpid())])
torch.manual_seed(0); np.random.seed(0)

## 1. Data: decode scans and masks once

Same colour decoding as the exploration notebook (nearest reference colour). Blended edge pixels
that match no colour are marked **ignore** (255): the model is neither rewarded nor penalised there.

In [ ]:
PALETTE = np.array([(0, 0, 0), (255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)], dtype=np.int32)

def decode(mask_rgb):
    d = ((mask_rgb[..., None, :].astype(np.int32) - PALETTE) ** 2).sum(-1)
    lab = d.argmin(-1).astype(np.uint8)
    lab[np.sqrt(d.min(-1)) > 120] = IGNORE
    return lab

def read_split(fn):
    return [Path(l.strip()).name for l in open(ROOT / fn) if l.strip()]

eye_of = lambda n: n.split("_")[0]
names_all = read_split("training.txt") + read_split("validation.txt")
split_of = {n: "train" for n in read_split("training.txt")} | {n: "val" for n in read_split("validation.txt")}
names_all = [n for n in names_all if (IMG_DIR / eye_of(n) / n).exists()]
if QUICK_TEST:
    tr = [n for n in names_all if split_of[n] == "train"][:48]
    va = [n for n in names_all if split_of[n] == "val"][:16]
    names_all = tr + va

cache_file = CACHE / ("data_quick.npz" if QUICK_TEST else "data.npz")
if cache_file.exists():
    z = np.load(cache_file, allow_pickle=False)
    names, imgs, masks = list(z["names"]), z["imgs"], z["masks"]
    print("loaded cache:", cache_file)
else:
    imgs = np.zeros((len(names_all), H, W_USE), np.uint8)
    masks = np.zeros((len(names_all), H, W_USE), np.uint8)
    t0 = time.time()
    for i, n in enumerate(names_all):
        a = np.asarray(Image.open(IMG_DIR / eye_of(n) / n).convert("RGB"))
        imgs[i] = a[:, :W_USE].mean(-1).astype(np.uint8)
        masks[i] = decode(a[:, W_PANEL:W_PANEL + W_USE])
        if (i + 1) % 500 == 0:
            print(f"{i + 1}/{len(names_all)} ({time.time() - t0:.0f}s)")
    names = names_all
    np.savez_compressed(cache_file, names=np.array(names), imgs=imgs, masks=masks)
    print("saved cache:", cache_file)

splits = np.array([split_of[n] for n in names])
eyes = np.array([eye_of(n) for n in names])
print(f"{len(names)} scans: {(splits == 'train').sum()} train, {(splits == 'val').sum()} val")

**Choosing the best epoch without touching the test set.** The official validation split is what we
evaluate denoising on, so it must not be used to pick the checkpoint. About 10% of the *training eyes*
are held out as a **tuning set** for that instead.

In [ ]:
rng = np.random.default_rng(0)
train_eyes = np.unique(eyes[splits == "train"])
tune_eyes = set(rng.choice(train_eyes, max(1, round(0.1 * len(train_eyes))), replace=False))
idx_train = np.where((splits == "train") & ~np.isin(eyes, list(tune_eyes)))[0]
idx_tune = np.where((splits == "train") & np.isin(eyes, list(tune_eyes)))[0]
idx_val = np.where(splits == "val")[0]
print(f"fit: {len(idx_train)} scans | tune: {len(idx_tune)} scans ({len(tune_eyes)} eyes) | val (test): {len(idx_val)} scans")

MEAN = float(imgs[idx_train].mean()) / 255
STD = float(imgs[idx_train].std()) / 255
print(f"intensity mean {MEAN:.3f}, std {STD:.3f}")

def to_input(batch_u8):
    """uint8 (B, 380, 558) -> normalised float tensor (B, 1, 384, 560), reflect-padded."""
    x = torch.from_numpy(np.ascontiguousarray(batch_u8)).float().div(255).sub(MEAN).div(STD)[:, None]
    return F.pad(x, (0, PAD_W - W_USE, 0, PAD_H - H), mode="reflect")

def to_target(batch_u8):
    y = torch.from_numpy(np.ascontiguousarray(batch_u8)).long()
    return F.pad(y, (0, PAD_W - W_USE, 0, PAD_H - H), value=IGNORE)

## 2. Model: a standard U-Net

Encoder-decoder with skip connections, which keep full resolution for small structures.
Six outputs (background, SRF, IRF, PED, SHRM, ellipsoid zone).

In [ ]:
class Block(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, n_classes=len(CLASSES), base=32):
        super().__init__()
        ch = [base, base * 2, base * 4, base * 8, base * 16]
        self.downs = nn.ModuleList([Block(1, ch[0])] + [Block(ch[i], ch[i + 1]) for i in range(4)])
        self.ups = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in reversed(range(4))])
        self.decs = nn.ModuleList([Block(ch[i] * 2, ch[i]) for i in reversed(range(4))])
        self.head = nn.Conv2d(ch[0], n_classes, 1)

    def forward(self, x):
        skips = []
        for i, down in enumerate(self.downs):
            x = down(x if i == 0 else F.max_pool2d(x, 2))
            skips.append(x)
        x = skips.pop()
        for up, dec in zip(self.ups, self.decs):
            x = dec(torch.cat([up(x), skips.pop()], 1))
        return self.head(x)

BASE = 32
model = UNet(base=BASE).to(DEVICE)
print(f"U-Net parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

## 3. Train on the original scans

Loss = cross-entropy + Dice (Dice helps the rare, small classes). Light augmentation: horizontal
flips and brightness/contrast changes. After every epoch the model is scored on the tuning set
(mean Dice over the four lesion classes) and saved:

- `seg_checkpoints/last.pt`: to resume if interrupted
- `seg_checkpoints/best.pt`: the epoch with the best tuning Dice, used for evaluation

To retrain from scratch, delete the `seg_checkpoints` folder.
If training stops with an out-of-memory error, set `BATCH = 2` below and delete `seg_checkpoints`.

In [ ]:
EPOCHS = 2 if QUICK_TEST else 40
BATCH = 4      # about 3 GB of memory; lower to 2 if you get an out-of-memory error
LR = 1e-3

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS,
                                            steps_per_epoch=int(np.ceil(len(idx_train) / BATCH)))

def dice_loss(logits, y):
    valid = (y != IGNORE)
    p = logits.softmax(1)
    oh = F.one_hot(torch.where(valid, y, 0), len(CLASSES)).permute(0, 3, 1, 2).float()
    v = valid[:, None].float()
    inter = (p * oh * v).sum((0, 2, 3))
    denom = (p * v).sum((0, 2, 3)) + (oh * v).sum((0, 2, 3))
    return 1 - ((2 * inter + 1) / (denom + 1))[1:].mean()

def augment(xb_u8):
    x = xb_u8.astype(np.float32)
    gain = np.random.uniform(0.85, 1.15, (len(x), 1, 1)); bias = np.random.uniform(-12, 12, (len(x), 1, 1))
    return np.clip(x * gain + bias, 0, 255).astype(np.uint8)

@torch.no_grad()
def predict(batch_u8):
    """uint8 scans (B, 380, 558) -> predicted labels (B, 380, 558)."""
    model.eval()
    out = []
    for i in range(0, len(batch_u8), 4):
        logits = model(to_input(batch_u8[i:i + 4]).to(DEVICE))
        out.append(logits.argmax(1)[:, :H, :W_USE].cpu().numpy().astype(np.uint8))
    return np.concatenate(out)

def lesion_dice(pred, gt):
    """Mean Dice over lesion classes, pooled over all pixels of the set."""
    d = []
    for c in LESION_IDS:
        p, g = pred == c, gt == c
        if g.sum() + p.sum():
            d.append(2 * (p & g).sum() / (p.sum() + g.sum()))
    return float(np.mean(d))

start_epoch, best, log = 0, -1.0, []
if (CKPT / "last.pt").exists():
    ck = torch.load(CKPT / "last.pt", map_location=DEVICE, weights_only=False)
    if ck.get("quick") == QUICK_TEST and ck.get("epochs") == EPOCHS:
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        start_epoch, best, log = ck["epoch"] + 1, ck["best"], ck["log"]
        print(f"resuming after epoch {ck['epoch'] + 1}; best tuning Dice so far {best:.3f}")
    else:
        print("last.pt is from a different setting (QUICK_TEST/EPOCHS): starting fresh")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    t0, losses = time.time(), []
    perm = np.random.permutation(idx_train)
    for b in range(0, len(perm), BATCH):
        ib = perm[b:b + BATCH]
        xb, yb = imgs[ib], masks[ib]
        flip = np.random.rand(len(ib)) < 0.5
        xb[flip], yb[flip] = xb[flip][:, :, ::-1], yb[flip][:, :, ::-1]
        x, y = to_input(augment(xb)).to(DEVICE), to_target(yb).to(DEVICE)
        logits = model(x)
        loss = F.cross_entropy(logits, y, ignore_index=IGNORE) + dice_loss(logits, y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        losses.append(loss.item())
    tune_dice = lesion_dice(predict(imgs[idx_tune]), masks[idx_tune])
    log.append({"epoch": epoch + 1, "loss": float(np.mean(losses)), "tune_dice": tune_dice, "sec": time.time() - t0})
    if tune_dice > best:
        best = tune_dice
        torch.save({"model": model.state_dict(), "epoch": epoch, "tune_dice": best, "mean": MEAN, "std": STD,
                    "base": BASE, "quick": QUICK_TEST}, CKPT / "best.pt")
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "epoch": epoch,
                "best": best, "log": log, "quick": QUICK_TEST, "epochs": EPOCHS}, CKPT / "last.pt")
    print(f"epoch {epoch + 1}/{EPOCHS}  loss {log[-1]['loss']:.3f}  tuning lesion Dice {tune_dice:.3f}"
          f"{'  (best)' if tune_dice == best else ''}  {log[-1]['sec']:.0f}s")

log_df = pd.DataFrame(log)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(log_df.epoch, log_df.loss); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
ax[1].plot(log_df.epoch, log_df.tune_dice); ax[1].set_title("tuning-set lesion Dice"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

In [ ]:
ck = torch.load(CKPT / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ck["model"]); MEAN, STD = ck["mean"], ck["std"]
MODEL_TAG = f"unet{ck['base']}_ep{ck['epoch'] + 1}" + ("_quick" if ck["quick"] else "")
print(f"using best checkpoint: epoch {ck['epoch'] + 1}, tuning lesion Dice {ck['tune_dice']:.3f}  ->  tag {MODEL_TAG}")

# Sanity check on the original validation scans
val_pred = predict(imgs[idx_val])
print("Dice per class on original validation scans:")
for c in LESIONS + ["ISOS"]:
    k = CLASSES.index(c); p, g = val_pred == k, masks[idx_val] == k
    print(f"  {c:5s} {2 * (p & g).sum() / max(p.sum() + g.sum(), 1):.3f}")

In [ ]:
COLORS = PALETTE.astype(np.uint8)
def colour(lab):
    out = np.zeros(lab.shape + (3,), np.uint8)
    for k in range(len(CLASSES)):
        out[lab == k] = COLORS[k]
    return out

show = idx_val[np.linspace(0, len(idx_val) - 1, 3).astype(int)]
fig, axes = plt.subplots(3, 3, figsize=(15, 8))
for r, i in enumerate(show):
    j = list(idx_val).index(i)
    axes[r, 0].imshow(imgs[i], cmap="gray"); axes[r, 0].set_title(names[i])
    axes[r, 1].imshow(colour(masks[i])); axes[r, 1].set_title("expert mask")
    axes[r, 2].imshow(colour(val_pred[j])); axes[r, 2].set_title("U-Net prediction")
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## 4. Conditions to compare

Found automatically in `denoised_cache/`:
- `original`: the scan as acquired
- `bm3d_<sigma>`: the BM3D files from the exploration notebook (`<scan>_s<sigma>.png`)
- any sub-folder `denoised_cache/<method>/<scan>.png`: a new denoiser

Only scans available in **every** condition are evaluated, so each comparison is on identical scans.

In [ ]:
def find_conditions():
    conds = {"original": None}
    if DENOISED.exists():
        sig = sorted({f.stem.rsplit("_s", 1)[1] for f in DENOISED.glob("*_s*.png")}, key=float)
        for s in sig:
            conds[f"bm3d_{s}"] = ("bm3d", s)
        for d in sorted(p for p in DENOISED.iterdir() if p.is_dir()):
            conds[d.name] = ("folder", d)
    return conds

def denoised_path(cond_spec, name):
    stem = Path(name).stem
    if cond_spec[0] == "bm3d":
        return DENOISED / f"{stem}_s{cond_spec[1]}.png"
    return cond_spec[1] / f"{stem}.png"

def load_condition(cond_spec, i):
    if cond_spec is None:
        return imgs[i]
    a = np.asarray(Image.open(denoised_path(cond_spec, names[i])).convert("L"))
    return a[:H, :W_USE]

CONDITIONS = find_conditions()
eval_idx = [i for i in idx_val
            if all(spec is None or denoised_path(spec, names[i]).exists() for spec in CONDITIONS.values())]
bm3d_conds = [c for c in CONDITIONS if c.startswith("bm3d_")]
STRONGEST = bm3d_conds[-1] if bm3d_conds else list(CONDITIONS)[-1]
print("conditions:", list(CONDITIONS), "| strongest BM3D:", STRONGEST)
print(f"{len(eval_idx)} of {len(idx_val)} validation scans are available in every condition")

## 5. Evaluate every condition

For every expert-marked lesion (>= 20 px) in every evaluated scan:
- **detected**: at least 10% of its pixels are predicted as the right class
- **coverage**: fraction of its pixels predicted as the right class
- **area ratio**: predicted area of that class around the lesion / true area (1 = correct,
  below 1 = measured too small, above 1 = too large)

And per scan and class:
- **invented lesions**: predicted patches (>= 20 px) that touch no expert lesion of that class
- **Dice** and **total area error** (predicted minus true area of each fluid type: what a clinician tracks)

Results are written per condition under `seg_eval/<model tag>/`, so finished conditions are skipped
when you rerun, and results from a retrained model never mix with old ones.

In [ ]:
EVAL = Path("seg_eval") / MODEL_TAG; EVAL.mkdir(parents=True, exist_ok=True)
STRUCT = np.ones((3, 3))

def score_scan(pred, gt):
    lesion_rows, scan_rows = [], []
    for c in LESIONS:
        k = CLASSES.index(c)
        g, p = gt == k, pred == k
        comp, n = ndimage.label(g, STRUCT)
        halo = ndimage.binary_dilation(g, iterations=2)
        if n:
            areas = ndimage.sum(g, comp, range(1, n + 1))
            dil = [ndimage.binary_dilation(comp == j, iterations=3) for j in range(1, n + 1)]
            for j in range(n):
                area = int(areas[j])
                if area < MIN_LESION_PX:
                    continue
                les = comp == j + 1
                others = np.zeros_like(g)
                for q in range(n):
                    if q != j:
                        others |= comp == q + 1
                cov = (p & les).sum() / area
                region = dil[j] & ~others
                lesion_rows.append({"class": c, "area_px": area, "coverage": cov, "detected": cov >= 0.10,
                                    "area_ratio": (p & region).sum() / area})
        pc, pn = ndimage.label(p, STRUCT)
        invented = 0
        if pn:
            pareas = ndimage.sum(p, pc, range(1, pn + 1))
            touched = set(np.unique(pc[halo & p])) - {0}
            invented = sum(1 for j in range(pn) if pareas[j] >= MIN_LESION_PX and (j + 1) not in touched)
        scan_rows.append({"class": c, "gt_px": int(g.sum()), "pred_px": int(p.sum()),
                          "inter_px": int((p & g).sum()), "invented": invented})
    return lesion_rows, scan_rows

for cond, spec in CONDITIONS.items():
    les_f, scan_f = EVAL / f"lesions_{cond}.csv", EVAL / f"scans_{cond}.csv"
    if les_f.exists() and scan_f.exists():
        print(f"{cond}: already evaluated, skipping"); continue
    t0, L, S = time.time(), [], []
    for b in range(0, len(eval_idx), 16):
        ib = eval_idx[b:b + 16]
        preds = predict(np.stack([load_condition(spec, i) for i in ib]))
        for i, pr in zip(ib, preds):
            lr, sr = score_scan(pr, masks[i])
            meta = {"image": names[i], "eye": eyes[i], "condition": cond}
            L += [meta | r for r in lr]; S += [meta | r for r in sr]
    pd.DataFrame(L).to_csv(les_f, index=False); pd.DataFrame(S).to_csv(scan_f, index=False)
    print(f"{cond}: {len(L)} lesions scored in {time.time() - t0:.0f}s")

lesions = pd.concat([pd.read_csv(EVAL / f"lesions_{c}.csv", dtype={"eye": str}) for c in CONDITIONS])
scans = pd.concat([pd.read_csv(EVAL / f"scans_{c}.csv", dtype={"eye": str}) for c in CONDITIONS])
COND_ORDER = list(CONDITIONS)
print(len(lesions) // len(CONDITIONS), "lesions per condition,", lesions["eye"].nunique(), "eyes")

## 6. Results

### 6a. Detection rate by lesion size

Share of expert lesions the model still finds. If denoising hurts small lesions, the small-lesion
row will drop under strong denoising while the large-lesion rows stay flat.

In [ ]:
BINS = [20, 50, 200, 1000, 5000, np.inf]
BIN_LABELS = ["20-50", "50-200", "200-1k", "1k-5k", ">5k"]
lesions["size_bin"] = pd.cut(lesions["area_px"], BINS, labels=BIN_LABELS, right=False)

def by_size(col, agg="mean"):
    t = lesions.pivot_table(index="size_bin", columns="condition", values=col, aggfunc=agg, observed=False)
    return t[COND_ORDER]

det = by_size("detected").round(3)
det["n_lesions"] = lesions[lesions.condition == "original"].groupby("size_bin", observed=False).size()
print("Detection rate (share of expert lesions found)")
det

In [ ]:
print(f"Detection rate by class and size, original vs {STRONGEST}")
strongest = STRONGEST
for c in LESIONS:
    sub = lesions[(lesions["class"] == c) & lesions.condition.isin(["original", strongest])]
    t = sub.pivot_table(index="size_bin", columns="condition", values="detected", aggfunc="mean", observed=False)
    t = t.reindex(columns=[x for x in ["original", strongest] if x in t.columns])
    n = sub[sub.condition == "original"].groupby("size_bin", observed=False).size()
    print(f"\n{c}"); print(t.assign(n=n).round(2).to_string())

### 6b. Measured size: are lesions traced too small or too large?

Median area ratio (predicted area around the lesion / true area). Below 1 = the model measures the
lesion smaller than it is. This matters for fluid monitoring even when the lesion is still detected.

In [ ]:
area = by_size("area_ratio", "median").round(2)
print("Median area ratio (1 = correct size)")
area

### 6c. Invented lesions, Dice and total fluid area per scan

- **invented per scan**: lesion-sized patches predicted where the expert marked none of that class
- **Dice**: pixel overlap with the expert mask (pooled over scans)
- **total area error**: median of (predicted - true) / true total area of that fluid type per scan,
  for scans that contain at least 50 px of it

In [ ]:
rows = []
for cond in COND_ORDER:
    s = scans[scans.condition == cond]
    for c in LESIONS:
        sc = s[s["class"] == c]
        has = sc[sc.gt_px >= 50]
        rows.append({"condition": cond, "class": c,
                     "invented_per_scan": sc.invented.mean(),
                     "dice": 2 * sc.inter_px.sum() / max(sc.pred_px.sum() + sc.gt_px.sum(), 1),
                     "total_area_error": ((has.pred_px - has.gt_px) / has.gt_px).median()})
summary = pd.DataFrame(rows)
for col in ["invented_per_scan", "dice", "total_area_error"]:
    print(f"\n{col}")
    print(summary.pivot(index="class", columns="condition", values=col)[COND_ORDER].round(3).to_string())

### 6d. The key comparison, with confidence intervals

How much does denoising lower detection for **small** lesions (< 200 px) compared with **large**
ones (>= 1000 px)? `extra loss for small` = (drop for small) - (drop for large). Intervals come from
resampling whole eyes. If the interval excludes 0, small lesions are hit harder, beyond chance.

In [ ]:
orig = lesions[lesions.condition == "original"].reset_index(drop=True)
small_mask = orig["area_px"] < 200
large_mask = orig["area_px"] >= 1000
eye_ids = orig["eye"].unique()
rng_b = np.random.default_rng(42)
pos_by_eye = {e: np.where(orig["eye"].to_numpy() == e)[0] for e in eye_ids}

def drops(det_o, det_d, pick):
    s, l = small_mask.to_numpy()[pick], large_mask.to_numpy()[pick]
    ds = det_o[pick][s].mean() - det_d[pick][s].mean()
    dl = det_o[pick][l].mean() - det_d[pick][l].mean()
    return ds, dl, ds - dl

rows = []
det_o = orig["detected"].to_numpy().astype(float)
for cond in COND_ORDER[1:]:
    d = lesions[lesions.condition == cond].reset_index(drop=True)
    assert len(d) == len(orig)
    det_d = d["detected"].to_numpy().astype(float)
    full = np.arange(len(orig))
    ds, dl, gap = drops(det_o, det_d, full)
    boots = []
    for _ in range(1000):
        pick = np.concatenate([pos_by_eye[e] for e in rng_b.choice(eye_ids, len(eye_ids), replace=True)])
        boots.append(drops(det_o, det_d, pick)[2])
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    rows.append({"condition": cond, "drop small": round(ds, 3), "drop large": round(dl, 3),
                 "extra loss for small": round(gap, 3), "95% CI": f"[{lo:.3f}, {hi:.3f}]",
                 "CI excludes 0": bool(lo > 0 or hi < 0)})
key = pd.DataFrame(rows)
key.to_csv(EVAL / "key_comparison.csv", index=False)
print(f"small lesions: {small_mask.sum()}, large lesions: {large_mask.sum()}, eyes: {len(eye_ids)}")
key

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
centres = [35, 100, 450, 2200, 9000]
for ax, tab, title, ylab in [(axes[0], det, "Detection rate", "share of lesions found"),
                             (axes[1], area, "Measured size", "median area ratio (1 = correct)")]:
    for cond in COND_ORDER:
        ax.plot(centres, tab[cond], marker="o", lw=2, label=cond)
    ax.set_xscale("log"); ax.set_xticks(centres); ax.set_xticklabels(BIN_LABELS)
    ax.set_xlabel("lesion size (pixels)"); ax.set_ylabel(ylab); ax.set_title(title); ax.legend(fontsize=8)
axes[1].axhline(1, color="grey", ls="--", lw=1)
inv = summary.groupby("condition")["invented_per_scan"].sum()[COND_ORDER]
axes[2].bar(range(len(inv)), inv.values, color="#4a6fa5")
axes[2].set_xticks(range(len(inv))); axes[2].set_xticklabels(inv.index, rotation=30, ha="right")
axes[2].set_title("Invented lesions per scan (all classes)")
plt.tight_layout(); plt.savefig(EVAL / "segmentation_by_size.png", dpi=150); plt.show()

### 6e. Look at small lesions lost under the strongest denoising

Small lesions the model found on the original scan but missed after the strongest denoising.
Columns: original | denoised | expert mask | prediction on original | prediction on denoised.

In [ ]:
if len(COND_ORDER) > 1:
    strongest = STRONGEST
    o = lesions[lesions.condition == "original"].reset_index(drop=True)
    d = lesions[lesions.condition == strongest].reset_index(drop=True)
    lost = o[(o.area_px < 200) & o.detected & ~d.detected].drop_duplicates("image").head(4)
    print(f"{((o.area_px < 200) & o.detected & ~d.detected).sum()} small lesions found on the original but missed with {strongest}")
    if len(lost):
        pos = {n: k for k, n in enumerate(names)}
        fig, axes = plt.subplots(len(lost), 5, figsize=(19, 3.2 * len(lost)), squeeze=False)
        for r, (_, row) in enumerate(lost.iterrows()):
            i = pos[row.image]
            den = load_condition(CONDITIONS[strongest], i)
            p0, p1 = predict(np.stack([imgs[i], den]))
            for c, (img, t) in enumerate([(imgs[i], "original"), (den, strongest), (colour(masks[i]), "expert"),
                                          (colour(p0), "pred: original"), (colour(p1), f"pred: {strongest}")]):
                axes[r, c].imshow(img, cmap="gray" if img.ndim == 2 else None)
                axes[r, c].set_title(f"{row.image}  {t}" if c == 0 else t, fontsize=9); axes[r, c].axis("off")
        plt.tight_layout(); plt.show()

### How to read the results

- **6a / 6d** answer H2 on the clinical measure: does strong denoising make the model *miss*
  small lesions more than large ones? `CI excludes 0 = True` for the strongest BM3D and `False`
  at the noise-level sigma would match the contrast result.
- **6b** checks the other possible harm: lesions still found, but measured smaller.
- **6c** checks the opposite failure: blotches from strong denoising mistaken for fluid.
- **Caveat:** the model was trained on original scans only. A denoised scan looks different from
  anything it learned on, so part of any drop may come from that mismatch rather than lost lesions.
  The fair follow-up is to also train a model on denoised scans and compare.